# Practice 100 — Matching & Propensity Scores

**Theoretical context**: see `CLAUDE.md` in this folder before starting.

**Phases**: this notebook mirrors the phases in `CLAUDE.md` § Instructions.
Each phase's exercise calls into a `src/_0N_<phase_name>.py` companion module —
read that module's `TODO(human)` block before implementing it there, then
re-run the corresponding cell below.

**Centerpiece**: the LaLonde (1986) replication. `src/datasets.py` loads two
real datasets from the `causaldata` package: the randomized NSW job-training
experiment (`nsw_mixtape`, both arms randomized — a known-good benchmark ATE)
and the CPS-1 observational comparison group (`cps_mixtape`) LaLonde used to
show how badly a naive observational comparison fails outside a randomized
experiment.

## Setup

In [ ]:
import sys
from pathlib import Path

# Jupyter sets the kernel's cwd to this notebook's folder, so the practice root --
# where the `src` package lives -- is not on sys.path. Put it there.
_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import numpy as np
import pandas as pd
import xy.pyplot as plt

from src.datasets import COVARIATES, OUTCOME, load_lalonde
from src.plotting import estimate_comparison_plot, love_plot, overlap_density_plot

## Phase 1 — Propensity score estimation

We load the LaLonde setup: an experimental NSW benchmark, and an observational
comparison (NSW treated vs. CPS-1 controls) from a very different population.
The naive difference-in-means on the observational data is wildly off from the
experimental ATE — that gap is exactly what propensity-score methods try to close.

In [ ]:
data = load_lalonde()
obs = data.observational
X = obs[COVARIATES].to_numpy()
treat = obs["treat"].to_numpy()
y = obs[OUTCOME].to_numpy()

naive_diff = obs.loc[treat == 1, OUTCOME].mean() - obs.loc[treat == 0, OUTCOME].mean()
print(f"Experimental ATE (ground truth):      {data.tau_experimental:,.0f}")
print(f"Naive observational diff (NSW vs CPS): {naive_diff:,.0f}")
obs.head()

### Exercise — `src/_01_propensity_score.py :: fit_propensity_score`

Open `src/_01_propensity_score.py`, read the `TODO(human)` block above the
function, implement it there (not in this cell), then re-run the cell below.

In [ ]:
from src._01_propensity_score import fit_propensity_score

fit = fit_propensity_score(X, treat)
print(f"Propensity scores: min={fit.scores.min():.3f} max={fit.scores.max():.3f} mean={fit.scores.mean():.3f}")

fig = overlap_density_plot(fit.scores[treat == 1], fit.scores[treat == 0], title="Phase 1 — propensity overlap (pre-adjustment)")
fig

## Phase 2 — Covariate balance

The propensity score is only useful if it actually balances the covariates.
We compute the standardized mean difference (SMD) for every covariate on the
raw, unadjusted observational sample — expect large imbalances here.

### Exercise — `src/_02_balance.py :: standardized_mean_diff`

Open `src/_02_balance.py`, read the `TODO(human)` block above the function,
implement it there, then re-run the cell below.

In [ ]:
from src._02_balance import balance_table

smd_before = balance_table(X, treat)
for name, val in zip(COVARIATES, smd_before):
    print(f"{name:10s} SMD = {val:+.3f}")

## Phase 3 — Nearest-neighbor / caliper matching

Each treated unit is paired with the control unit that looks most like it on
the propensity score, discarding pairs too far apart (the caliper). The ATT
over matched pairs should land much closer to the experimental ATE than the
naive diff-in-means did.

### Exercise — `src/_03_matching.py :: nearest_neighbor_match`

Open `src/_03_matching.py`, read the `TODO(human)` block above the function,
implement it there, then re-run the cell below.

In [ ]:
from src._03_matching import nearest_neighbor_match

ps_t, ps_c = fit.scores[treat == 1], fit.scores[treat == 0]
y_t, y_c = y[treat == 1], y[treat == 0]
match_result = nearest_neighbor_match(ps_t, ps_c, y_t, y_c)
n_matched = int((match_result.matched_control_idx >= 0).sum())
print(f"Matched {n_matched}/{len(ps_t)} treated units")
print(f"Matching ATT: {match_result.att:,.0f}  (experimental ATE = {data.tau_experimental:,.0f})")

## Phase 4 — Inverse probability weighting (IPW)

Instead of discarding unmatched units, IPW reweights every unit by the inverse
of its probability of receiving the treatment arm it actually got.

### Exercise — `src/_04_ipw.py :: ipw_ate`

Open `src/_04_ipw.py`, read the `TODO(human)` block above the function,
implement it there, then re-run the cell below.

In [ ]:
from src._04_ipw import ipw_ate

tau_ipw = ipw_ate(y, treat, fit.scores)
print(f"IPW ATE: {tau_ipw:,.0f}  (experimental ATE = {data.tau_experimental:,.0f})")

ipw_weights = treat / fit.scores + (1 - treat) / (1 - fit.scores)
smd_after_ipw = balance_table(X, treat, weights=ipw_weights)
fig = love_plot(COVARIATES, smd_before, smd_after_ipw, after_label="after IPW", title="Phase 4 — balance before vs. after IPW")
fig

## Phase 5 — Doubly robust (AIPW) estimation

AIPW combines an outcome regression with the IPW correction term, staying
consistent if *either* model is right — hedging against Phase 4's dependence
on a correctly specified propensity model alone.

### Exercise — `src/_05_aipw.py :: aipw_ate`

Open `src/_05_aipw.py`, read the `TODO(human)` block above the function,
implement it there, then re-run the cell below.

In [ ]:
from src._05_aipw import aipw_ate, fit_outcome_regressions

mu1, mu0 = fit_outcome_regressions(X, treat, y)
tau_aipw = aipw_ate(y, treat, fit.scores, mu1, mu0)
print(f"AIPW ATE: {tau_aipw:,.0f}  (experimental ATE = {data.tau_experimental:,.0f})")

## Phase 6 — End-to-End Run: the LaLonde replication

Every method's estimate on the *same* observational comparison, next to the
experimental benchmark. Bootstrap CIs give each estimate a rough error bar.

In [ ]:
def bootstrap_ci(estimator_fn, n_boot=200, seed=0):
    rng = np.random.default_rng(seed)
    n = len(y)
    boots = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boots[b] = estimator_fn(idx)
    return np.percentile(boots, [2.5, 97.5])

def naive_fn(idx):
    t, yy = treat[idx], y[idx]
    return yy[t == 1].mean() - yy[t == 0].mean()

def ipw_fn(idx):
    return ipw_ate(y[idx], treat[idx], fit.scores[idx])

labels = ["experimental", "naive observational", "matching (ATT)", "IPW", "AIPW"]
estimates = [data.tau_experimental, naive_diff, match_result.att, tau_ipw, tau_aipw]
ci_lo, ci_hi = [], []
for label, point, fn in zip(labels, estimates, [None, naive_fn, None, ipw_fn, None]):
    if fn is None:
        # Matching/AIPW/experimental: report the point estimate with no CI band (width 0)
        # to keep this end-to-end cell's plumbing simple — the point comparison is the focus.
        ci_lo.append(point)
        ci_hi.append(point)
    else:
        lo, hi = bootstrap_ci(fn)
        ci_lo.append(lo)
        ci_hi.append(hi)

fig = estimate_comparison_plot(labels, estimates, ci_lo, ci_hi)
fig

In [ ]:
smd_after_match = balance_table(
    np.vstack([X[treat == 1], X[treat == 0][match_result.matched_control_idx[match_result.matched_control_idx >= 0]]]),
    np.concatenate([
        np.ones(int((match_result.matched_control_idx >= 0).sum())),
        np.zeros(int((match_result.matched_control_idx >= 0).sum())),
    ]),
)
fig = love_plot(COVARIATES, smd_before, smd_after_match, after_label="after matching", title="Phase 6 — balance before vs. after matching")
fig

## Verification

Sanity-checks that must pass once every TODO is implemented.

In [ ]:
# The naive observational estimate should be badly biased relative to the
# experimental benchmark — the classic LaLonde failure this practice reproduces.
assert abs(naive_diff - data.tau_experimental) > 5000, "naive observational estimate should be badly biased"

# Every adjustment method should land closer to the experimental ATE than the naive estimate did.
for label, est in zip(["matching", "IPW", "AIPW"], [match_result.att, tau_ipw, tau_aipw]):
    assert abs(est - data.tau_experimental) < abs(naive_diff - data.tau_experimental), (
        f"{label} should recover the experimental ATE better than the naive estimate"
    )

# IPW-weighted balance should improve (on average, in absolute value) over the raw imbalance.
assert np.mean(np.abs(smd_after_ipw)) < np.mean(np.abs(smd_before)), "IPW should improve average covariate balance"

print("OK")